# 4. Neural Network with Embedding Layer (BiLSTM + CNN)
**Dataset:** CLINC150 — 151-class intent classification

**Approach:** Trainable word embeddings → parallel BiLSTM + CNN branches → intent classification

**Expected Accuracy:** ~88–92%

**Requirements:** `pip install torch`

### Imports

In [7]:
import os, re, collections
import numpy as np
import pandas as pd
from datasets import load_dataset

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
torch.manual_seed(42)

Using device: cpu


### Load Dataset

In [8]:
test_dataset  = load_dataset('parquet', data_files={'test':       os.path.join('..', 'data', 'test-00000-of-00001.parquet')})
train_dataset = load_dataset('parquet', data_files={'train':      os.path.join('..', 'data', 'train-00000-of-00001.parquet')})
val_dataset   = load_dataset('parquet', data_files={'validation': os.path.join('..', 'data', 'validation-00000-of-00001.parquet')})

train_df = train_dataset['train'].to_pandas()
val_df   = val_dataset['validation'].to_pandas()
test_df  = test_dataset['test'].to_pandas()

NUM_CLASSES = train_df['intent'].nunique()
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Classes: {NUM_CLASSES}')

Train: 10625 | Val: 3100 | Test: 5500 | Classes: 151


### Vocabulary & Tokenizer

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

MAX_LEN  = 32   # CLINC150 utterances are short
MIN_FREQ = 2
PAD_TOKEN, UNK_TOKEN = '<PAD>', '<UNK>'

def preprocess(sentences):
    base_stop_words = set(stopwords.words('english'))

    words_to_keep = {'what', 'where', 'who', 'why', 'how', 'when', 'do'}

    custom_stop_words = base_stop_words - words_to_keep

    allowed_symbols = {'?', '!'}

    lemmatizer = WordNetLemmatizer()
    result = []
    for sent in sentences:
        tokens = word_tokenize(sent)
        pos_tags = nltk.pos_tag(tokens)
        cleaned = []
        for token, pos in pos_tags:
            t = token.lower()
            if (t not in custom_stop_words) and (t.isalpha() or t in allowed_symbols):
                if t.isalpha():
                    if pos.startswith('NN'):   lemma = lemmatizer.lemmatize(t, 'n')
                    elif pos.startswith('VB'): lemma = lemmatizer.lemmatize(t, 'v')
                    elif pos.startswith('JJ'): lemma = lemmatizer.lemmatize(t, 'a')
                    else:                      lemma = lemmatizer.lemmatize(t)
                else:
                    lemma = t
                cleaned.append(lemma)
        result.append(' '.join(cleaned))
    return result

# Build vocabulary from training set only
counter = collections.Counter()
for sent in train_df['text']:
    counter.update(preprocess([sent]))

vocab = [PAD_TOKEN, UNK_TOKEN] + [w for w, c in counter.most_common() if c >= MIN_FREQ]
word2idx = {w: i for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)
PAD_IDX    = word2idx[PAD_TOKEN]
UNK_IDX    = word2idx[UNK_TOKEN]
print(f'Vocabulary size: {VOCAB_SIZE}')

def encode(text: str, max_len: int = MAX_LEN):
    tokens = preprocess(text)
    ids    = [word2idx.get(t, UNK_IDX) for t in tokens[:max_len]]
    # Pad to max_len
    ids   += [PAD_IDX] * (max_len - len(ids))
    return ids

Vocabulary size: 2381


### Dataset & DataLoader

In [4]:
class IntentDataset(Dataset):
    def __init__(self, df):
        self.X = [encode(t) for t in df['text']]
        self.y = df['intent'].tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.long)
        )

BATCH_SIZE = 64
train_loader = DataLoader(IntentDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(IntentDataset(val_df),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(IntentDataset(test_df),  batch_size=BATCH_SIZE)

print(f'Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

Batches — Train: 167 | Val: 49 | Test: 86


### Model Architecture — BiLSTM + CNN (parallel)

```
Input tokens
    │
 Embedding (300-d)
    ├─── BiLSTM (256 hidden) ──► last hidden state (512-d)
    └─── Conv1D (128 filters, kernels 3,4,5) ──► MaxPool ──► concat (384-d)
                           │
                        Concat (512+384 = 896-d)
                           │
                        Dropout(0.4) → FC(256) → ReLU → Dropout → FC(151)
```

In [ ]:
class IntentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_filters,
                 kernel_sizes, num_classes, pad_idx, dropout=0.4):
        super().__init__()

        # Shared embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        nn.init.xavier_uniform_(self.embedding.weight.data)
        self.embedding.weight.data[pad_idx].fill_(0)

        # BiLSTM branch
        self.bilstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=2,
            batch_first=True, bidirectional=True,
            dropout=dropout
        )

        # CNN branch — parallel filters
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=k//2)
            for k in kernel_sizes
        ])

        lstm_out_dim = hidden_dim * 2          # bidirectional
        cnn_out_dim  = num_filters * len(kernel_sizes)
        combined_dim = lstm_out_dim + cnn_out_dim

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(combined_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        emb = self.embedding(x)                   # (B, T, E)

        # BiLSTM: take final hidden from last layer
        _, (h_n, _) = self.bilstm(emb)
        # h_n: (num_layers*2, B, hidden) → concat last layer fwd+bwd
        lstm_out = torch.cat([h_n[-2], h_n[-1]], dim=-1)  # (B, hidden*2)

        # CNN: conv over sequence dim
        emb_t = emb.permute(0, 2, 1)              # (B, E, T) for Conv1d
        cnn_out = torch.cat([
            torch.max(torch.relu(conv(emb_t)), dim=-1).values
            for conv in self.convs
        ], dim=-1)                                 # (B, F*len(kernels))

        combined = torch.cat([lstm_out, cnn_out], dim=-1)
        return self.classifier(combined)


EMBED_DIM    = 300
HIDDEN_DIM   = 256
NUM_FILTERS  = 128
KERNEL_SIZES = [3, 4, 5]

model = IntentClassifier(
    vocab_size   = VOCAB_SIZE,
    embed_dim    = EMBED_DIM,
    hidden_dim   = HIDDEN_DIM,
    num_filters  = NUM_FILTERS,
    kernel_sizes = KERNEL_SIZES,
    num_classes  = NUM_CLASSES,
    pad_idx      = PAD_IDX,
    dropout      = 0.5
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')
print(model)

Trainable parameters: 1,934,227
IntentClassifier(
  (embedding): Embedding(2381, 300, padding_idx=0)
  (bilstm): LSTM(300, 128, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (convs): ModuleList(
    (0): Conv1d(300, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): Conv1d(300, 64, kernel_size=(4,), stride=(1,), padding=(2,))
    (2): Conv1d(300, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  )
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=448, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=256, out_features=151, bias=True)
  )
)


### Training Loop

In [6]:
EPOCHS    = 30
LR        = 1e-3
PATIENCE  = 5

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=2, factor=0.5)

best_val_acc   = 0.0
patience_count = 0

def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(X_batch)
            preds  = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return accuracy_score(all_labels, all_preds)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        total_loss += loss.item()

    val_acc = evaluate(val_loader)
    scheduler.step(val_acc)
    print(f'Epoch {epoch:02d}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc*100:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc   = val_acc
        patience_count = 0
        torch.save(model.state_dict(), 'best_bilstm_cnn.pt')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nBest Val Accuracy: {best_val_acc*100:.2f}%')

Epoch 01/30 | Loss: 4.6223 | Val Acc: 16.52%
Epoch 02/30 | Loss: 2.7647 | Val Acc: 42.06%
Epoch 03/30 | Loss: 1.6282 | Val Acc: 57.81%
Epoch 04/30 | Loss: 1.0724 | Val Acc: 67.45%
Epoch 05/30 | Loss: 0.7241 | Val Acc: 73.39%
Epoch 06/30 | Loss: 0.5405 | Val Acc: 79.19%
Epoch 07/30 | Loss: 0.4038 | Val Acc: 79.74%
Epoch 08/30 | Loss: 0.3313 | Val Acc: 82.19%
Epoch 09/30 | Loss: 0.2751 | Val Acc: 82.42%
Epoch 10/30 | Loss: 0.2463 | Val Acc: 83.19%
Epoch 11/30 | Loss: 0.2079 | Val Acc: 82.74%
Epoch 12/30 | Loss: 0.1812 | Val Acc: 84.81%
Epoch 13/30 | Loss: 0.1696 | Val Acc: 84.16%
Epoch 14/30 | Loss: 0.1647 | Val Acc: 83.87%
Epoch 15/30 | Loss: 0.1350 | Val Acc: 83.74%
Epoch 16/30 | Loss: 0.1017 | Val Acc: 84.84%
Epoch 17/30 | Loss: 0.0859 | Val Acc: 84.84%
Epoch 18/30 | Loss: 0.0783 | Val Acc: 84.94%
Epoch 19/30 | Loss: 0.0689 | Val Acc: 84.58%
Epoch 20/30 | Loss: 0.0600 | Val Acc: 84.58%
Epoch 21/30 | Loss: 0.0640 | Val Acc: 84.16%
Epoch 22/30 | Loss: 0.0469 | Val Acc: 84.77%
Epoch 23/3

### Evaluation on Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_bilstm_cnn.pt', map_location=DEVICE))

test_acc = evaluate(test_loader)
print(f'Test Accuracy: {test_acc*100:.2f}%')

# Full report
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        preds   = torch.argmax(model(X_batch), dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.numpy())

print(classification_report(all_labels, all_preds))

### Inference Helper

In [ ]:
intent_mapping = {
    0: 'restaurant_reviews', 1: 'nutrition_info', 2: 'account_blocked', 3: 'oil_change_how', 4: 'time', 
    5: 'weather', 6: 'redeem_rewards', 7: 'interest_rate', 8: 'gas_type', 9: 'accept_reservations', 
    10: 'smart_home', 11: 'user_name', 12: 'report_lost_card', 13: 'repeat', 14: 'whisper_mode', 
    15: 'what_are_your_hobbies', 16: 'order', 17: 'jump_start', 18: 'schedule_meeting', 19: 'meeting_schedule', 
    20: 'freeze_account', 21: 'what_song', 22: 'meaning_of_life', 23: 'restaurant_reservation', 24: 'traffic', 
    25: 'make_call', 26: 'text', 27: 'bill_balance', 28: 'improve_credit_score', 29: 'change_language', 
    30: 'no', 31: 'measurement_conversion', 32: 'timer', 33: 'flip_coin', 34: 'do_you_have_pets', 
    35: 'balance', 36: 'tell_joke', 37: 'last_maintenance', 38: 'exchange_rate', 39: 'uber', 
    40: 'car_rental', 41: 'credit_limit', 42: 'oos', 43: 'shopping_list', 44: 'expiration_date', 
    45: 'routing', 46: 'meal_suggestion', 47: 'tire_change', 48: 'todo_list', 49: 'card_declined', 
    50: 'rewards_balance', 51: 'change_accent', 52: 'vaccines', 53: 'reminder_update', 54: 'food_last', 
    55: 'change_ai_name', 56: 'bill_due', 57: 'who_do_you_work_for', 58: 'share_location', 59: 'international_visa', 
    60: 'calendar', 61: 'translate', 62: 'carry_on', 63: 'book_flight', 64: 'insurance_change', 
    65: 'todo_list_update', 66: 'timezone', 67: 'cancel_reservation', 68: 'transactions', 69: 'credit_score', 
    70: 'report_fraud', 71: 'spending_history', 72: 'directions', 73: 'spelling', 74: 'insurance', 
    75: 'what_is_your_name', 76: 'reminder', 77: 'where_are_you_from', 78: 'distance', 79: 'payday', 
    80: 'flight_status', 81: 'find_phone', 82: 'greeting', 83: 'alarm', 84: 'order_status', 
    85: 'confirm_reservation', 86: 'cook_time', 87: 'damaged_card', 88: 'reset_settings', 89: 'pin_change', 
    90: 'replacement_card_duration', 91: 'new_card', 92: 'roll_dice', 93: 'income', 94: 'taxes', 
    95: 'date', 96: 'who_made_you', 97: 'pto_request', 98: 'tire_pressure', 99: 'how_old_are_you', 
    100: 'rollover_401k', 101: 'pto_request_status', 102: 'how_busy', 103: 'application_status', 104: 'recipe', 
    105: 'calendar_update', 106: 'play_music', 107: 'yes', 108: 'direct_deposit', 109: 'credit_limit_change', 
    110: 'gas', 111: 'pay_bill', 112: 'ingredients_list', 113: 'lost_luggage', 114: 'goodbye', 
    115: 'what_can_i_ask_you', 116: 'book_hotel', 117: 'are_you_a_bot', 118: 'next_song', 119: 'change_speed', 
    120: 'plug_type', 121: 'maybe', 122: 'w2', 123: 'oil_change_when', 124: 'thank_you', 125: 'shopping_list_update', 
    126: 'pto_balance', 127: 'order_checks', 128: 'travel_alert', 129: 'fun_fact', 130: 'sync_device', 
    131: 'schedule_maintenance', 132: 'apr', 133: 'transfer', 134: 'ingredient_substitution', 135: 'calories', 
    136: 'current_location', 137: 'international_fees', 138: 'calculator', 139: 'definition', 140: 'next_holiday', 
    141: 'update_playlist', 142: 'mpg', 143: 'min_payment', 144: 'change_user_name', 145: 'restaurant_suggestion', 
    146: 'travel_notification', 147: 'cancel', 148: 'pto_used', 149: 'travel_suggestion', 150: 'change_volume'
}


In [ ]:
def predict_intent(text: str) -> str:
    model.eval()
    ids = torch.tensor([encode(text)], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = model(ids)
    pred_id = torch.argmax(logits, dim=-1).item()
    return intent_mapping.get(pred_id, str(pred_id))

for sample in ['what is the weather today', 'book a flight to London', 'tell me a joke']:
    print(f'{sample!r:45s} → {predict_intent(sample)}')